# 四色积木连续识别与中心坐标

联调演示，不控制舵机，不是正式 detect 服务。
先解压配套 infer_project_om.tar.gz 到开发板 RobotArm 目录。需要板端 CANN、ais_bench、PyTorch 等课程环境。
图像坐标来自原始摄像头画面，没有透视校正或手眼标定。
每次运行采集100帧，阈值0.4仅用于演示；正式接口默认0.5，由部署模块执行。
重复运行会复用 live_model。不要在同一内核调用 finalize 后再次加载模型。全部测试结束时关闭内核。

In [ ]:
import os
import sys
import time

PROJECT = os.environ.get("JXB_INFER_PROJECT", "/home/HwHiAiUser/RobotArm/infer_project")
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import cv2
import numpy as np
import torch
from IPython.display import display, Image
from ais_bench.infer.interface import InferSession
from om_infer import preprocess_img_batch, read_class_names, draw_bbox
from edge_infer.det_utils import letterbox, scale_coords, nms

# 同一个内核只加载一次，重复运行时复用模型
if "live_model" not in globals():
    live_model = InferSession(0, "output/yolov5s_bs1.om")

names = read_class_names("test/test.json")
screen = display("", display_id=True)
cap = None

try:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("摄像头打开失败")

    for i in range(100):
        ok, frame = cap.read()
        if not ok or frame is None:
            raise RuntimeError("摄像头读取失败")

        # 图像预处理
        padded, ratio, pad = letterbox(
            frame, new_shape=(640, 640)
        )
        inputs = preprocess_img_batch(
            np.expand_dims(padded, axis=0)
        )

        # 模型推理
        outputs = live_model.infer([inputs])
        boxes = nms(
            torch.tensor(outputs[0]),
            conf_thres=0.4,
            iou_thres=0.5
        )[0].numpy()

        # 将检测框还原到摄像头画面坐标
        scale_coords(
            (640, 640),
            boxes[:, :4],
            frame.shape,
            ratio_pad=(ratio, pad)
        )

        # 绘制检测框与颜色
        shown = draw_bbox(
            boxes, frame.copy(), (0, 255, 0), 2, names
        )
        cv2.putText(
            shown,
            f"Frame {i + 1}/100",
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2
        )

        # 更新 Notebook 中的画面
        encoded, jpg = cv2.imencode(".jpg", shown)
        if encoded:
            screen.update(Image(data=jpg.tobytes()))

        time.sleep(0.05)

except KeyboardInterrupt:
    print("已手动停止")

finally:
    if cap is not None:
        cap.release()

print("测试结束，摄像头已释放；模型保留，可再次运行此代码框。")

## 最后一帧的中心坐标
先执行上一个单元格。本单元格不拍新照片，不用于实时抓取。

In [ ]:
center_view = frame.copy()
detections = []

for x1, y1, x2, y2, confidence, class_id in boxes:
    u = float((x1 + x2) / 2)
    v = float((y1 + y2) / 2)
    color = names[int(class_id)]

    detections.append({
        "color": color.lower(),
        "confidence": round(float(confidence), 3),
        "center_u_px": round(u, 1),
        "center_v_px": round(v, 1),
    })

    cv2.rectangle(
        center_view,
        (int(x1), int(y1)), (int(x2), int(y2)),
        (0, 255, 0), 2
    )
    cv2.circle(center_view, (round(u), round(v)), 5, (0, 0, 255), -1)
    cv2.putText(
        center_view,
        f"{color} ({u:.0f}, {v:.0f})",
        (int(x1), max(20, int(y1) - 8)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1
    )

encoded, jpg = cv2.imencode(".jpg", center_view)
if encoded:
    display(Image(data=jpg.tobytes()))

for detection in detections:
    print(detection)